In [1]:
import pandas as pd
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By

# ==============================
# 1. KHỞI TẠO SELENIUM
# ==============================

driver = webdriver.Chrome()

base_url = "https://www.cgv.vn"
url = "https://www.cgv.vn/default/movies/coming-soon-1.html"

driver.get(url)
time.sleep(4)


# ==============================
# 2. LẤY DANH SÁCH PHIM SẮP CHIẾU
# ==============================

soup = BeautifulSoup(driver.page_source, "html.parser")

movies = []

# Các thẻ phim trên trang Coming Soon
movie_items = soup.select("li.film-lists.item")

print("Số phim tìm thấy:", len(movie_items))

for item in movie_items:

    # Tên + link phim
    title_tag = item.select_one("h2.product-name a")

    if not title_tag:
        continue

    movie_name = title_tag.get_text(" ", strip=True)
    movie_url = title_tag.get("href")

    if movie_url and movie_url.startswith("/"):
        movie_url = base_url + movie_url

    # Thể loại
    genre = ""

    info_items = item.select("div.cgv-movie-info")

    for info in info_items:
        text = info.get_text(" ", strip=True)

        if "Thể loại:" in text:
            genre = text.replace("Thể loại:", "").strip()

    # Thời lượng
    duration = ""

    for info in info_items:
        text = info.get_text(" ", strip=True)

        if "Thời lượng:" in text:
            duration = text.replace("Thời lượng:", "").strip()

    # Khởi chiếu
    release_date = ""

    for info in info_items:
        text = info.get_text(" ", strip=True)

        if "Khởi chiếu:" in text:
            release_date = text.replace("Khởi chiếu:", "").strip()

    movies.append({
        "MovieName": movie_name,
        "MovieURL": movie_url,
        "Genre": genre,
        "ReleaseDate": release_date,
        "Duration": duration
    })


print("\nDanh sách phim:")
for movie in movies:
    print(movie["MovieName"])

Số phim tìm thấy: 34

Danh sách phim:
TRÁI TIM QUÁI THÚ
THẦN SƯ CHUNG QUỲ: LINH GIỚI ĐẠI CHIẾN
KHÓA CHẶT CỬA NÀO SUZUME
QUYẾT CUA ANH NÀY
ÁN MẠNG KARAOKE
QUỶ ĂN TẠNG 4: HỔ TINH
ÁN MẠNG XÉM HOÀN HẢO
CHUYỆN CÔNG CHÚA KAGUYA
NGƯỜI MẸ KHÁC
ALWAYS LALISA
CHỊ CHỊ EM EM 3
NGỌN ĐỒI HOA HỒNG ANH
PHIM STREET FIGHTER
DIGGER
CHUYỆN TÌNH KHAU VAI
MẸ MÌN
CLAYFACE
TRẤN YỂM
BÒ SỮA BAY
GODZILLA TRỪ KHÔNG
CHÀNG MÈO MANG MŨ
NGÀY CON CÒN MẸ
EBENEZER PHÉP MÀU ĐÊM GIÁNG SINH
TRẠI GIAM HẠNH PHÚC
GẶP GỠ THÔNG GIA: DÂU MỚI TRÌNH LÀNG
SỢI CHỈ ĐỎ
DUNE: HÀNH TINH CÁT - PHẦN BA
AVENGERS: NGÀY TẬN THẾ
THÁM TỬ KIÊN: LỜI NGUYỀN HOÀNG KIM
LOẠN THẾ
ÁC TƯỢNG
QUỶ DẮT TAY
CHIẾN BÀO
CỘNG TRỪ NHÂN CHIA


In [2]:
# ==============================
# 3. CÀO CHI TIẾT TỪNG PHIM
# ==============================

upcoming_movies = []

for i, movie in enumerate(movies):

    print(f"[{i+1}/{len(movies)}] Đang cào: {movie['MovieName']}")

    try:
        driver.get(movie["MovieURL"])
        time.sleep(2)

        soup = BeautifulSoup(driver.page_source, "html.parser")

        data = {
            "MovieName": movie["MovieName"],
            "MovieURL": movie["MovieURL"],
            "Rating": "",
            "Director": "",
            "Actors": "",
            "Genre": "",
            "ReleaseDate": "",
            "Duration": "",
            "Language": ""
        }

        # ==========================
        # LẤY CÁC movie-info
        # ==========================

        info_blocks = soup.select("div.movie-info")

        for block in info_blocks:

            label = block.select_one("label")
            value = block.select_one(".std")

            if not label or not value:
                continue

            field = label.get_text(" ", strip=True)
            value_text = value.get_text(" ", strip=True)

            if "Đạo diễn" in field:
                data["Director"] = value_text

            elif "Diễn viên" in field:
                data["Actors"] = value_text

            elif "Thể loại" in field:
                data["Genre"] = value_text

            elif "Khởi chiếu" in field:
                data["ReleaseDate"] = value_text

            elif "Thời lượng" in field:
                data["Duration"] = value_text

            elif "Ngôn ngữ" in field:
                data["Language"] = value_text

        # ==========================
        # LẤY RATING
        # ==========================

        rated_text = soup.get_text(" ", strip=True)

        if "Rated:" in rated_text:

            start = rated_text.find("Rated:")

            rating_text = rated_text[start:start + 100]

            # Lấy phần đầu sau Rated:
            rating_text = rating_text.replace("Rated:", "").strip()

            # Chỉ lấy mã rating đầu tiên
            rating_parts = rating_text.split()

            if len(rating_parts) > 0:
                data["Rating"] = rating_parts[0]

        upcoming_movies.append(data)

    except Exception as e:

        print("Lỗi:", e)

        # Vẫn giữ lại thông tin cơ bản
        upcoming_movies.append({
            "MovieName": movie["MovieName"],
            "MovieURL": movie["MovieURL"],
            "Rating": "",
            "Director": "",
            "Actors": "",
            "Genre": movie.get("Genre", ""),
            "ReleaseDate": movie.get("ReleaseDate", ""),
            "Duration": movie.get("Duration", ""),
            "Language": ""
        })


# Đóng trình duyệt
driver.quit()

print("\nĐã cào xong!")
print("Số phim:", len(upcoming_movies))

[1/34] Đang cào: TRÁI TIM QUÁI THÚ
[2/34] Đang cào: THẦN SƯ CHUNG QUỲ: LINH GIỚI ĐẠI CHIẾN
[3/34] Đang cào: KHÓA CHẶT CỬA NÀO SUZUME
[4/34] Đang cào: QUYẾT CUA ANH NÀY
[5/34] Đang cào: ÁN MẠNG KARAOKE
[6/34] Đang cào: QUỶ ĂN TẠNG 4: HỔ TINH
[7/34] Đang cào: ÁN MẠNG XÉM HOÀN HẢO
[8/34] Đang cào: CHUYỆN CÔNG CHÚA KAGUYA
[9/34] Đang cào: NGƯỜI MẸ KHÁC
[10/34] Đang cào: ALWAYS LALISA
[11/34] Đang cào: CHỊ CHỊ EM EM 3
[12/34] Đang cào: NGỌN ĐỒI HOA HỒNG ANH
[13/34] Đang cào: PHIM STREET FIGHTER
[14/34] Đang cào: DIGGER
[15/34] Đang cào: CHUYỆN TÌNH KHAU VAI
[16/34] Đang cào: MẸ MÌN
[17/34] Đang cào: CLAYFACE
[18/34] Đang cào: TRẤN YỂM
[19/34] Đang cào: BÒ SỮA BAY
[20/34] Đang cào: GODZILLA TRỪ KHÔNG
[21/34] Đang cào: CHÀNG MÈO MANG MŨ
[22/34] Đang cào: NGÀY CON CÒN MẸ
[23/34] Đang cào: EBENEZER PHÉP MÀU ĐÊM GIÁNG SINH
[24/34] Đang cào: TRẠI GIAM HẠNH PHÚC
[25/34] Đang cào: GẶP GỠ THÔNG GIA: DÂU MỚI TRÌNH LÀNG
[26/34] Đang cào: SỢI CHỈ ĐỎ
[27/34] Đang cào: DUNE: HÀNH TINH CÁT - PHẦN BA
[2

In [4]:
df_upcoming = pd.DataFrame(upcoming_movies)


In [5]:
df_upcoming.to_csv(
    r"D:\Project_Group3\cgv_upcoming_movies.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu: cgv_upcoming_movies.csv")

Đã lưu: cgv_upcoming_movies.csv
